# Token Contextual Analysis

How token representations change based on context: sensitivity, neighbor
influence, divergence, contextual embeddings, and uniqueness.

## Why This Matters

Token contextual tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_contextual_analysis import (
    token_context_sensitivity, token_neighbor_influence,
    token_representation_divergence, token_contextual_embedding,
    token_unique_information,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Context Sensitivity

How much does context change a token's representation vs its raw embedding?

In [ ]:
result = token_context_sensitivity(model, tokens)
print(f"Token {result['token_id']} at position {result['position']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['context_influence'] * 20)
    print(f"  Layer {p['layer']}: cos_to_embed={p['cosine_to_embed']:.4f}, "
          f"context_influence={p['context_influence']:.4f} {bar}")

## Neighbor Influence

How much attention does the last position pay to each source?

In [ ]:
result = token_neighbor_influence(model, tokens)
for s in result['per_source']:
    bar = '█' * int(s['fraction'] * 40)
    print(f"  Pos {s['position']} (tok {s['token_id']:3d}): "
          f"attn={s['total_attention']:.3f} ({s['fraction']:.1%}) {bar}")

## Representation Divergence

Do token representations become more diverse (different) in deeper layers?

In [ ]:
result = token_representation_divergence(model, tokens)
for p in result['per_layer']:
    bar = '█' * int(p['diversity'] * 20)
    print(f"  Layer {p['layer']}: pairwise_cos={p['mean_pairwise_cosine']:.4f}, "
          f"diversity={p['diversity']:.4f} {bar}")

## Contextual Embedding

How each token's final representation relates to its original embedding.

In [ ]:
result = token_contextual_embedding(model, tokens, layer=-1)
for t in result['per_token']:
    print(f"  Pos {t['position']} (tok {t['token_id']:3d}): "
          f"cos={t['cosine_to_embed']:.4f}, shift={t['context_shift']:.4f}, "
          f"norm {t['embed_norm']:.3f}→{t['contextual_norm']:.3f}")

## Unique Information

How much unique information does a position carry (distance to nearest neighbor)?

In [ ]:
result = token_unique_information(model, tokens, position=-1)
for p in result['per_layer']:
    bar = '█' * int(p['uniqueness'] * 20)
    print(f"  Layer {p['layer']}: nearest_cos={p['nearest_cosine']:.4f} "
          f"(pos {p['nearest_position']}), uniqueness={p['uniqueness']:.4f} {bar}")